In [1]:
import cv2
import time
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Video, display

In [2]:
model = YOLO("yolo11n.pt")

print("YOLO11n model loaded successfully!")

YOLO11n model loaded successfully!


In [3]:
# Base project directory (handles running from inside notebooks/ or root)
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Input video path
video_path = PROJECT_DIR / "dataset" / "video.mp4"

# Output directory
output_dir = PROJECT_DIR / "runs" / "person_detection_video"

# Create output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)

# Output video path
output_path = output_dir / "output.mp4"

print("Project root:", PROJECT_DIR)
print("Input video:", video_path)
print("Output video:", output_path)

Input video: dataset/video.mp4
Output video: runs/person_detection_video/output.mp4


In [4]:
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise FileNotFoundError(
        f"Unable to open video: {video_path}. "
        "Please add video.mp4 inside the dataset folder."
    )

# Read video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if fps <= 0:
    fps = 25

duration = total_frames / fps if total_frames > 0 else 0

print("Video opened successfully!")
print("Width:", width)
print("Height:", height)
print("FPS:", fps)
print("Total frames:", total_frames)
print("Duration:", round(duration, 2), "seconds")

OpenCV: Couldn't read video stream from file "dataset/video.mp4"


FileNotFoundError: Unable to open video: dataset/video.mp4. Please add video.mp4 inside the dataset folder.

In [5]:
# Define video codec
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

# Initialize output video writer
out = cv2.VideoWriter(
    str(output_path),
    fourcc,
    fps,
    (width, height)
)

if not out.isOpened():
    cap.release()
    raise RuntimeError("Unable to initialize output video writer")

print("Output video writer initialized successfully!")

NameError: name 'fps' is not defined

In [ ]:
frame_count = 0
total_detection_time = 0

while True:

    # Read next frame
    ret, frame = cap.read()

    # Stop when video ends
    if not ret:
        break

    # Start processing timer
    start_time = time.time()

    # Perform YOLO person detection
    results = model.predict(
        source=frame,
        classes=[0],
        conf=0.25,
        imgsz=640,
        device="cpu",
        verbose=False
    )

    # Calculate detection time
    detection_time = time.time() - start_time
    total_detection_time += detection_time

    # Draw bounding boxes
    annotated_frame = results[0].plot()

    # Save processed frame
    out.write(annotated_frame)

    frame_count += 1

    # Display progress
    if frame_count % 100 == 0:
        print(f"Processed {frame_count} frames")

print("Frame processing completed!")

In [ ]:
cap.release()
out.release()

print("Video resources released successfully!")

In [ ]:
average_detection_time = (
    total_detection_time / frame_count
    if frame_count > 0 else 0
)

average_fps = (
    frame_count / total_detection_time
    if total_detection_time > 0 else 0
)

print("========== VIDEO PROCESSING SUMMARY ==========")

print("Total Frames Processed:", frame_count)
print("Original Video FPS:", round(fps, 2))
print("Average Detection Time:", round(average_detection_time, 4), "seconds")
print("Average Processing FPS:", round(average_fps, 2))
print("Output Video:", output_path)